# Notebook 2) Rock Classification

## Workflow

1. **Data Loading and Preprocessing**: Load modal mineral abundance data from Excel file and normalize mineral abundances and filter samples
2. **Model Training and Evaluation**: Train multiple classification models (Rule-based, GNB, SVM, LR, RF, MLP, XGBoost) and evaluate models on test set and save trained models
3. **Inference**: Load trained models and perform end-to-end classification on SEM images

## Classification Task

- **Input**: Modal mineral abundances (Ol, Py, Pl, Ms, Si, Op) as proportions
- **Output**: Rock type classification
  - Class 0: Ilmenite Basalt (IB)
  - Class 1: Olivine Basalt (OB)
  - Class 2: Pigeonite Basalt (PB)

In [1]:
# Setup for Google Colab environment
# Note: This cell is specific to Google Colab. For local execution, modify paths accordingly.
from google.colab import drive
drive.mount('/content/drive')

FOLDERNAME = 'JGRML-mineral-identification-2025'
assert FOLDERNAME is not None, "[!] Enter the foldername."

import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TORCH_USE_CUDA_DSA'] = '1'

!cd /content/drive/MyDrive/$FOLDERNAME

Mounted at /content/drive


# 1. Dataset Overview

## Excel Dataset Structure

The classification dataset is provided as an Excel file (`data-lsc-modal.xlsx`) containing modal mineral abundance data for lunar mare basalt samples. The dataset includes the following columns:

### Column Descriptions

- **ID** and **No**: Sample number and subsample designation, respectively. A paragraph symbol (¶) next to the ID indicates samples recognized as mare basalts, but no mineralogical mode information is provided in the available literature.

- **Type**: Rock classification labels:
  - **IB** = Ilmenite Basalt
  - **OB** = Olivine Basalt
  - **PB** = Pigeonite Basalt
  - **PlB** = Plagioclase Basalt
  - **FB** = Feldspathic Basalt
  - **KB** = KREEP Basalt
  - **U** = Unclassified
  
  A dagger (†) next to the rock type indicates that a different classification appears in the Lunar Sample Information Catalog.

- **Method**: Indicates how modal mineral abundances were obtained:
  - **O** = Optical (manual point counting)
  - **M** = Microprobe / automated EDS-based mode
  - **G** = Grain mount/crush–separate analysis
  - **B** = Binocular or hand-specimen description
  - **X** = Automated X-ray imaging
  - **C** = Bulk chemical analysis
  
  A section mark (§) after the method denotes additional notes provided in the spreadsheet.

- **Count**: Number of counted points, or **QE** for qualitative visual estimation. A double-dagger (‡) indicates that the modal estimate was reconstructed from the reported count area and grid spacing rather than directly published point counts.

- **Mineral Abundances**: Six mineral columns representing modal proportions:
  - **Ol** = Olivine
  - **Py** = Pyroxene
  - **Pl** = Plagioclase
  - **Ms** = Mesostasis or groundmass
  - **Si** = Silica-rich phases
  - **Op** = Opaque minerals (ilmenite, spinel, troilite, Fe–Ni metal)

- **Reference**: Source of the modal value. An asterisk (*) indicates that the modal value was taken from the *Lunar Sample Compendium*; either the primary cited source could not be located or the reported value does not appear in the original text.

### Data Preprocessing

For this classification task, we:
1. Filter samples to include only those with valid references (exclude entries with "Reference" = "-")
2. Focus on the three main rock types: IB, OB, and PB
3. Normalize mineral abundances to proportions (0-1 range) by dividing by the sum
4. Handle missing values and data type conversions
/

In [2]:
# Import required libraries for data processing and machine learning
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning libraries
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Suppress warnings from openpyxl (Excel file reading)
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl.worksheet._reader")

In [4]:
# Load and preprocess modal mineral abundance data from Excel file
# This data contains mineral proportions for each sample and corresponding rock type labels

file_path = f'/content/drive/MyDrive/{FOLDERNAME}/data/data-lsc-modal.xlsx'

# Load the Excel file
df = pd.read_excel(file_path, sheet_name='Sheet1')

# Initialize data cleaning
df.fillna(0, inplace=True)

# Define mineral columns (6 features for classification)
# Ol: Olivine, Py: Pyroxene, Pl: Plagioclase, Ms: Mesostasis, Si: Silicate, Op: Opaque
minerals = ["Ol", "Py", "Pl", "Ms", "Si", "Op"]

# Convert mineral columns to numeric, handling any conversion errors
for col in minerals:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(0)

# Normalize mineral abundances: convert absolute values to proportions
# Divide each mineral abundance by the sum to get proportions (0-1 range)
df["Type"] = df["Type"].replace({"IB†": "IB"})  # Fix typo in rock type label

for mineral in minerals:
    df[mineral] = df[mineral] / df["sum"]

# Filter data: keep only samples with valid references
df = df[df["Reference"] != "-"]

# Filter to keep only the three main rock types of interest
# IB: Ilmenite Basalt, OB: Olivine Basalt, PB: Pigeonite Basalt
df_filtered = df[df["Type"].isin(["IB", "OB", "PB"])]

# Display dataset statistics after filtering
unique_rock_types = df_filtered["Type"].unique()
rock_type_counts = df_filtered["Type"].value_counts()

print("Unique Rock Types after filtering:")
print(unique_rock_types)
print("\nRock Type Distribution:")
print(rock_type_counts)

# Encode rock type labels to numerical values for classification
# 0: Ilmenite Basalt (IB), 1: Olivine Basalt (OB), 2: Pigeonite Basalt (PB)
df_filtered = df_filtered.copy()
df_filtered.loc[:, 'Type Code'] = df_filtered['Type'].astype('category').cat.codes

# Extract features (mineral abundances) and labels (rock type codes)
features = df_filtered[["Ol", "Py", "Pl", "Ms", "Si", "Op"]]
labels = df_filtered['Type Code']

print(f"\nDataset Summary:")
print(f"  Total samples: {len(features)}")
print(f"  Features shape: {features.shape}")
print(f"  Feature columns: {list(features.columns)}")
print(f"\nFirst few samples:")
print(features.head())
print(f"\nLabel distribution:")
print(labels.value_counts().sort_index())

Unique Rock Types after filtering:
['IB' 'OB' 'PB']

Rock Type Distribution:
Type
IB    397
OB    147
PB    106
Name: count, dtype: int64

Dataset Summary:
  Total samples: 650
  Features shape: (650, 6)
  Feature columns: ['Ol', 'Py', 'Pl', 'Ms', 'Si', 'Op']

First few samples:
         Ol        Py        Pl        Ms        Si        Op
0  0.000000  0.521169  0.292339  0.000000  0.003024  0.183468
1  0.000000  0.486000  0.335000  0.008000  0.014000  0.157000
2  0.000000  0.497522  0.334985  0.013875  0.011893  0.141724
3  0.005145  0.494756  0.338413  0.010390  0.010489  0.140807
4  0.005010  0.487976  0.348697  0.000000  0.010020  0.148297

Label distribution:
Type Code
0    397
1    147
2    106
Name: count, dtype: int64


In [5]:
# Split dataset into training and test sets
# Using 90% for training and 10% for testing
# random_state ensures reproducibility of the split
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.1, random_state=22, stratify=labels
)

# Standardize features using StandardScaler
# This normalizes features to have zero mean and unit variance
# Important for distance-based and gradient-based algorithms (SVM, Logistic Regression, etc.)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Store feature order for later use (important for inference)
feature_order = list(features.columns)

print(f"Training set size: {len(X_train_scaled)}")
print(f"Test set size: {len(X_test_scaled)}")
print(f"Feature order: {feature_order}")
print(f"\nTraining set label distribution:")
print(pd.Series(y_train).value_counts().sort_index())
print(f"\nTest set label distribution:")
print(pd.Series(y_test).value_counts().sort_index())

Training set size: 585
Test set size: 65
Feature order: ['Ol', 'Py', 'Pl', 'Ms', 'Si', 'Op']

Training set label distribution:
Type Code
0    357
1    132
2     96
Name: count, dtype: int64

Test set label distribution:
Type Code
0    40
1    15
2    10
Name: count, dtype: int64


# 2. Model Training

This section trains and evaluates multiple classification models to predict rock types (Ilmenite Basalt, Olivine Basalt, Pigeonite Basalt) from modal mineral abundances. We compare a rule-based baseline classifier with several machine learning approaches.

## Classification Approaches

### 1. Rule-Based Baseline Classifier

A simple heuristic classifier that uses domain knowledge about mineral abundance patterns:

- **Olivine Basalt (OB)**: Identified when olivine abundance > 5%
- **Ilmenite Basalt (IB)**: Identified when opaque minerals > 7% and olivine ≤ 5%
- **Pigeonite Basalt (PB)**: Default classification for all other cases


### 2. Machine Learning Classifiers

We train and evaluate six different machine learning algorithms:

1. **Gaussian Naive Bayes (GNB)**
2. **Support Vector Machine (SVM)**
3. **Logistic Regression (LR)**
4. **Random Forest (RF)**
5. **Multilayer Perceptron (MLP)**
6. **XGBoost**

### Training Configuration

- **Train/Test Split**: 90% training, 10% testing (stratified to maintain class distribution)
- **Feature Scaling**: StandardScaler applied to normalize features (zero mean, unit variance)
- **Evaluation Metrics**: Accuracy, precision, recall, F1-score per class and overall
- **Model Persistence**: All trained models are saved as `.pkl` files for later inference


In [6]:
# Define class names and mapping for rock types
# This mapping is used for displaying human-readable class names in results

class_names = [
    "Ilmenite Basalt",    # Class 0
    "Olivine Basalt",     # Class 1
    "Pigeonite Basalt"    # Class 2
]

rock_type_mapping = dict(enumerate(class_names))

print("Rock Type Class Mapping:")
for code, name in rock_type_mapping.items():
    print(f"  {code}: {name}")

Rock Type Class Mapping:
  0: Ilmenite Basalt
  1: Olivine Basalt
  2: Pigeonite Basalt


In [7]:
# Import classification utilities from local module
# These functions provide:
# - rule_based_classifier: Heuristic classification based on mineral abundance rules
# - evaluate_rule_based: Evaluate rule-based classifier on dataset
# - evaluate_model: Train and evaluate a machine learning classifier
# - get_models: Get dictionary of configured ML classifiers
from utils.rockClassifier import (
    CLASS_NAMES,
    rule_based_classifier,
    evaluate_rule_based,
    evaluate_model,
    get_models
)

In [8]:
# Train and evaluate all classification models
# This cell trains multiple ML classifiers and saves them for later use

import joblib
import os

# Create directory for saving models
model_dir = f"/content/drive/MyDrive/{FOLDERNAME}/models/rockClassifiers"
os.makedirs(model_dir, exist_ok=True)

# ============================================
# 1. Evaluate Rule-Based Baseline Classifier
# ============================================
# The rule-based classifier uses simple heuristics:
# - Olivine basalt: olivine > 5%
# - Ilmenite basalt: opaques > 7% and olivine ≤ 5%
# - Otherwise: pigeonite basalt
print("=" * 60)
print("Evaluating Rule-Based Baseline Classifier")
print("=" * 60)
evaluate_rule_based(
    df=df_filtered,
    minerals=minerals,
    labels=labels
)

# ============================================
# 2. Train Machine Learning Classifiers
# ============================================
# Get dictionary of configured classifiers
# Models include: GNB, SVM, Logistic Regression, Random Forest, MLP, XGBoost
models = get_models(random_state=42)
trained_models = {}

print("\n" + "=" * 60)
print("Training Machine Learning Classifiers")
print("=" * 60)

for name, model in models.items():
    print(f"\nTraining {name}...")

    # Train and evaluate the model
    trained_model = evaluate_model(
        model=model,
        X_train=X_train_scaled,  # Use scaled features
        y_train=y_train,
        X_test=X_test_scaled,    # Use scaled features
        y_test=y_test,
        model_name=name
    )

    trained_models[name] = trained_model

    # Save trained model to disk
    # Convert model name to filename (lowercase, replace spaces/special chars)
    fname = name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    model_path = os.path.join(model_dir, f"{fname}.pkl")
    joblib.dump(trained_model, model_path)
    print(f"  Saved: {model_path}")

# Also save the scaler for later use in inference
scaler_path = os.path.join(model_dir, "scaler.pkl")
joblib.dump(scaler, scaler_path)
print(f"\nScaler saved: {scaler_path}")

print("\n" + "=" * 60)
print("All models saved successfully!")
print(f"Model directory: {model_dir}")
print("=" * 60)

Evaluating Rule-Based Baseline Classifier

###### Rule-Based Classifier ######
Accuracy: 0.81
                  precision    recall  f1-score   support

 Ilmenite Basalt       0.94      0.80      0.87       397
  Olivine Basalt       0.64      0.84      0.73       147
Pigeonite Basalt       0.72      0.80      0.76       106

        accuracy                           0.81       650
       macro avg       0.77      0.82      0.78       650
    weighted avg       0.84      0.81      0.82       650


Training Machine Learning Classifiers

Training Gaussian Naive Bayes...

###### Gaussian Naive Bayes (Train) ######
Accuracy: 0.88
                  precision    recall  f1-score   support

 Ilmenite Basalt       0.91      0.94      0.93       357
  Olivine Basalt       0.86      0.86      0.86       132
Pigeonite Basalt       0.79      0.69      0.73        96

        accuracy                           0.88       585
       macro avg       0.85      0.83      0.84       585
    weighted av

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


  Saved: /content/drive/MyDrive/JGRML-mineral-identification-2025/models/rockClassifiers/logistic_regression.pkl

Training Random Forest...

###### Random Forest (Train) ######
Accuracy: 1.00
                  precision    recall  f1-score   support

 Ilmenite Basalt       1.00      1.00      1.00       357
  Olivine Basalt       1.00      1.00      1.00       132
Pigeonite Basalt       1.00      0.99      0.99        96

        accuracy                           1.00       585
       macro avg       1.00      1.00      1.00       585
    weighted avg       1.00      1.00      1.00       585


###### Random Forest (Test) ######
Accuracy: 0.94
                  precision    recall  f1-score   support

 Ilmenite Basalt       0.97      0.93      0.95        40
  Olivine Basalt       0.93      0.93      0.93        15
Pigeonite Basalt       0.83      1.00      0.91        10

        accuracy                           0.94        65
       macro avg       0.91      0.95      0.93        6

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(



###### Multilayer Perceptron (Train) ######
Accuracy: 0.92
                  precision    recall  f1-score   support

 Ilmenite Basalt       0.95      0.95      0.95       357
  Olivine Basalt       0.88      0.86      0.87       132
Pigeonite Basalt       0.83      0.88      0.85        96

        accuracy                           0.92       585
       macro avg       0.89      0.89      0.89       585
    weighted avg       0.92      0.92      0.92       585


###### Multilayer Perceptron (Test) ######
Accuracy: 0.91
                  precision    recall  f1-score   support

 Ilmenite Basalt       0.97      0.95      0.96        40
  Olivine Basalt       0.86      0.80      0.83        15
Pigeonite Basalt       0.75      0.90      0.82        10

        accuracy                           0.91        65
       macro avg       0.86      0.88      0.87        65
    weighted avg       0.91      0.91      0.91        65

  Saved: /content/drive/MyDrive/JGRML-mineral-identification-20

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [12:47:32] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



###### XGBoost (Train) ######
Accuracy: 1.00
                  precision    recall  f1-score   support

 Ilmenite Basalt       1.00      1.00      1.00       357
  Olivine Basalt       1.00      1.00      1.00       132
Pigeonite Basalt       1.00      0.99      0.99        96

        accuracy                           1.00       585
       macro avg       1.00      1.00      1.00       585
    weighted avg       1.00      1.00      1.00       585


###### XGBoost (Test) ######
Accuracy: 0.92
                  precision    recall  f1-score   support

 Ilmenite Basalt       0.95      0.93      0.94        40
  Olivine Basalt       0.88      0.93      0.90        15
Pigeonite Basalt       0.90      0.90      0.90        10

        accuracy                           0.92        65
       macro avg       0.91      0.92      0.91        65
    weighted avg       0.92      0.92      0.92        65

  Saved: /content/drive/MyDrive/JGRML-mineral-identification-2025/models/rockClassifiers/xg

# 3. Model Inference on SEM Images

This section demonstrates how to use the trained classification models for end-to-end rock type classification on SEM images. The workflow is:

- **Load trained models**: Load segmentation model and classification models
- **Load test images**: Prepare SEM images for evaluation
- **Run inference**: For each image:
   - Segment minerals using UNet
   - Extract modal mineral abundances from segmentation
   - Classify rock type using trained classifiers
- **Evaluate results**: Compare predictions with ground truth and generate reports

## Prerequisites

- Trained segmentation model (UNet) saved as `.pth` file
- Trained classification models saved as `.pkl` files
- Test dataset with SEM images and ground truth labels

In [9]:
# Import libraries for image processing and model inference
import csv
import glob
import random
import joblib
import os

import cv2
import numpy as np
from PIL import Image

import matplotlib.colors as mcolors
from matplotlib import pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader, Dataset, Subset, random_split

from torchvision import datasets as tv_datasets
from torchvision import transforms as T

# Import local utility modules
from utils.loadData import *
from utils.unet import *
from utils.metrics import *
from utils.rockClassifier import *

In [10]:
# Configure device for PyTorch (GPU if available, otherwise CPU)
USE_GPU = True
dtype = torch.float32

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')
else:
    device = torch.device('cpu')
    print('Using CPU (GPU not available)')

print(f'Device: {device}')

Using GPU: NVIDIA A100-SXM4-40GB
Device: cuda


In [11]:
# Load trained classification models
# These models were trained in the previous section on modal mineral abundance data
model_dir = f"/content/drive/MyDrive/{FOLDERNAME}/models/rockClassifiers"

print("Loading trained classification models...")
gnb = joblib.load(os.path.join(model_dir, "gaussian_naive_bayes.pkl"))
svm = joblib.load(os.path.join(model_dir, "support_vector_machine_rbf.pkl"))
lr = joblib.load(os.path.join(model_dir, "logistic_regression.pkl"))
rf = joblib.load(os.path.join(model_dir, "random_forest.pkl"))
mlp = joblib.load(os.path.join(model_dir, "multilayer_perceptron.pkl"))
xgb = joblib.load(os.path.join(model_dir, "xgboost.pkl"))

# Load the scaler used during training (important for feature normalization)
scaler = joblib.load(os.path.join(model_dir, "scaler.pkl"))

print("All models loaded successfully!")
print(f"  - Gaussian Naive Bayes: ✓")
print(f"  - Support Vector Machine: ✓")
print(f"  - Logistic Regression: ✓")
print(f"  - Random Forest: ✓")
print(f"  - Multilayer Perceptron: ✓")
print(f"  - XGBoost: ✓")
print(f"  - StandardScaler: ✓")

Loading trained classification models...
All models loaded successfully!
  - Gaussian Naive Bayes: ✓
  - Support Vector Machine: ✓
  - Logistic Regression: ✓
  - Random Forest: ✓
  - Multilayer Perceptron: ✓
  - XGBoost: ✓
  - StandardScaler: ✓


In [12]:
# Load trained segmentation model (UNet)
# This model segments minerals from SEM images
# Note: Update the model path to match your trained segmentation model

in_channels = 1   # Grayscale input images
out_channels = 10 # Number of mineral classes
num_classes = out_channels

# Initialize UNet_1 model architecture
model = UNet_1(in_channels, out_channels).to(device)

# Load trained weights
# Note: Update this path to your saved segmentation model checkpoint
model_path = f"/content/drive/MyDrive/{FOLDERNAME}/models/Unets/m1-best-12800.pth"
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()  # Set to evaluation mode

print(f"Segmentation model loaded from: {model_path}")
print("Model set to evaluation mode.")

Segmentation model loaded from: /content/drive/MyDrive/JGRML-mineral-identification-2025/models/Unets/m1-best-12800.pth
Model set to evaluation mode.


In [13]:
# Extract test dataset
# Note: Update the path to match your test dataset location
import zipfile as zf

# Path to test dataset zip file
# For this example, we use a subset of the full dataset for evaluation
dataset_path = "/content/drive/MyDrive/{FOLDERNAME}/data/data-sem-label.zip"

files = zf.ZipFile(dataset_path, 'r')
files.extractall()
files.close()

print("Test dataset extracted successfully.")

Test dataset extracted successfully.


In [14]:
# Load test dataset directories
# The dataset contains SEM images with pixel scale information and ground truth labels

base_dir = 'data-sem-label'  # Root directory containing all data subfolders

# List all "input-features" directories (contain pixel scale and rock type text files)
input_features_folders = sorted([
    path for path in glob.glob(os.path.join(base_dir, '*', 'input-features'))
    if '.ipynb_checkpoints' not in path
])

# List all "input-images" directories (256 × 256 grayscale SEM input images)
input_image_folders = sorted([
    path for path in glob.glob(os.path.join(base_dir, '*', 'input-images'))
    if '.ipynb_checkpoints' not in path
])

# List all "output-images" directories (256 × 256 labeled segmentation masks)
output_image_folders = sorted([
    path for path in glob.glob(os.path.join(base_dir, '*', 'output-images'))
    if '.ipynb_checkpoints' not in path
])

# Define scale threshold for filtering (in micrometers)
# For this evaluation, we focus on large-scale images (> 1.8 μm)
scale_criteria = 1.8

print(f"Found {len(input_image_folders)} sample folders")
print(f"Scale threshold: {scale_criteria} μm (filtering for large-scale images)")

Found 28 sample folders
Scale threshold: 1.8 μm (filtering for large-scale images)


In [15]:
# Extract pixel size and ground truth rock type labels from text files
# Each text file contains pixel scale and rock type information

def get_pixel_size_and_label(filename):
    """
    Extract pixel size and rock type from a text file.

    Args:
        filename: Path to text file containing pixel_size and rock type information

    Returns:
        tuple: (pixel_size, rock_type) or (None, None) if not found
    """
    pixel_size = None
    rock_type = None
    try:
        with open(filename, 'r') as f:
            lines = f.readlines()
            for line in lines:
                if line.startswith("pixel_size:"):
                    pixel_size = float(line.split(":")[1].strip())
                elif line.startswith("rock type:"):
                    rock_type_str = line.split(":")[1].strip().lower()
                    # Map rock type strings to class indices
                    if "ilmenite" in rock_type_str:
                        rock_type = 0  # Ilmenite Basalt
                    elif "olivine" in rock_type_str:
                        rock_type = 1  # Olivine Basalt
                    elif "pigeonite" in rock_type_str:
                        rock_type = 2  # Pigeonite Basalt
    except FileNotFoundError:
        return None, None
    return pixel_size, rock_type

# Build dictionaries mapping image names to pixel sizes and ground truth labels
pixel_size_dict = {}
ground_truth_labels = {}

print("Extracting pixel sizes and ground truth labels...")
for input_features_folder in input_features_folders:
    for filename in os.listdir(input_features_folder):
        if filename.endswith(".txt"):
            base_name = filename.split('.txt')[0]
            pixel_size, rock_type = get_pixel_size_and_label(
                os.path.join(input_features_folder, filename)
            )

            # Filter: only keep large-scale images (> 1.8 μm) with valid labels
            if pixel_size is not None and rock_type is not None and pixel_size > scale_criteria:
                pixel_size_dict[base_name] = pixel_size
                ground_truth_labels[base_name] = rock_type

print(f"Found {len(pixel_size_dict)} images with pixel size > {scale_criteria} μm")
print(f"Ground truth label distribution:")
label_counts = pd.Series(list(ground_truth_labels.values())).value_counts().sort_index()
for label, count in label_counts.items():
    print(f"  Class {label} ({class_names[label]}): {count} samples")


Extracting pixel sizes and ground truth labels...
Found 8624 images with pixel size > 1.8 μm
Ground truth label distribution:
  Class 0 (Ilmenite Basalt): 2128 samples
  Class 1 (Olivine Basalt): 2800 samples
  Class 2 (Pigeonite Basalt): 3696 samples


In [16]:
# Organize test images into dataset
# Match input images with their corresponding output masks and filter by available labels

input_whole_pixel, output_whole_pixel, filenames_whole_pixel = [], [], []

for input_image_folder, output_image_folder in zip(input_image_folders, output_image_folders):
    for image_name in os.listdir(input_image_folder):
        if not image_name.endswith(".png"):
            continue

        pixel_key = os.path.splitext(image_name)[0]

        # Only include images that have pixel size and ground truth label information
        if pixel_key in pixel_size_dict:
            in_image_path = os.path.join(input_image_folder, image_name)
            out_image_path = os.path.join(output_image_folder, image_name)

            input_whole_pixel.append(in_image_path)
            output_whole_pixel.append(out_image_path)
            filenames_whole_pixel.append(image_name)

# Sample a subset of images for evaluation
# This is useful for quick testing. Set N to len(input_whole_pixel) for full evaluation
import random

N = 300  # Number of samples to evaluate (can be adjusted)
random.seed(42)  # For reproducibility

if len(input_whole_pixel) > N:
    idx = random.sample(range(len(input_whole_pixel)), N)
    input_whole_pixel = [input_whole_pixel[i] for i in idx]
    output_whole_pixel = [output_whole_pixel[i] for i in idx]
    filenames_whole_pixel = [filenames_whole_pixel[i] for i in idx]
    print(f"Sampled {N} images from {len(input_whole_pixel) + len(idx) - N} available images")
else:
    print(f"Using all {len(input_whole_pixel)} available images")

# Create PyTorch dataset for test images
datasets_whole_pixel = create_segmentation_dataset(
    input_whole_pixel,
    output_whole_pixel,
    pixel_size_dict,
    filenames_whole_pixel
)

print(f"\nTest dataset created: {len(datasets_whole_pixel)} images")

Sampled 300 images from 300 available images

Test dataset created: 300 images


In [17]:
# End-to-end evaluation: Segmentation + Classification
# This cell evaluates the complete pipeline on test images:
# 1. Segment minerals using UNet
# 2. Extract modal abundances from segmentation
# 3. Classify rock type using all trained classifiers
# 4. Compare with ground truth and generate reports

from tqdm.auto import tqdm
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np
import time
import traceback

# ============================================
# Setup
# ============================================
target_names = ['Ilmenite Basalt', 'Olivine Basalt', 'Pigeonite Basalt']
class_labels = [0, 1, 2]

# Initialize RockClassifier with feature order matching training data
# IMPORTANT: The feature_order must match the order used during training
# This ensures proper column alignment when transforming features
feature_order = ["Ol", "Py", "Pl", "Ms", "Si", "Op"]
classifier = RockClassifier(
    model, scaler, gnb, svm, lr, rf, mlp, xgb,
    pixel_size_dict, device,
    feature_order=feature_order
)

# Initialize lists to store predictions and ground truth
y_true = []
rb_pred, gnb_pred, svm_pred, lr_pred, rf_pred, mlp_pred, xgb_pred = [], [], [], [], [], [], []
image_ids = []
fail_log = []

N = len(datasets_whole_pixel)
assert N > 0, "Test dataset is empty. Check data loading."

print(f"Evaluating {N} test images...")
print("=" * 60)

# ============================================
# Evaluation Loop
# ============================================
# Process each image with progress bar showing running accuracy
# The progress bar displays real-time accuracy for each classifier as images are processed
pbar = tqdm(range(N), desc="Evaluating images", unit="img")

for idx in pbar:
    try:
        # Get sample from dataset
        sample = datasets_whole_pixel[idx]

        # Extract ground truth label
        # Dataset may return 4 or 5 elements depending on structure:
        # - 5 elements: (input, mask, pixel_size, label, filename)
        # - 4 elements: (input, mask, pixel_size, filename) - label from external mapping
        if len(sample) == 5:
            _, _, _, label_tensor, filename = sample
            true_label = int(label_tensor.item())
        elif len(sample) == 4:
            _, _, _, filename = sample
            base_tmp = os.path.splitext(filename)[0]
            # Use external ground truth mapping if available
            true_label = int(ground_truth_labels[base_tmp])
        else:
            raise RuntimeError(f"Unexpected sample length: {len(sample)}")

        base_name = os.path.splitext(filename)[0]
        input_image_path = datasets_whole_pixel.input_files[idx]

        # Run inference: segmentation + classification
        # Step 1: Load and preprocess image
        classifier.load_image(input_image_path)
        # Step 2: Run segmentation to get mineral class map
        classifier.predict()

        # Step 3: Extract modal abundances and classify using all methods
        rb_label = classifier.rb_modal_abundances(classifier.predicted_class_map_np)
        gnb_label = classifier.gnb_modal_abundances(classifier.predicted_class_map_np)
        svm_label = classifier.svm_modal_abundances(classifier.predicted_class_map_np)
        lr_label = classifier.lr_modal_abundances(classifier.predicted_class_map_np)
        rf_label = classifier.rf_modal_abundances(classifier.predicted_class_map_np)
        mlp_label = classifier.mlp_modal_abundances(classifier.predicted_class_map_np)
        xgb_label = classifier.xgb_modal_abundances(classifier.predicted_class_map_np)

        # Store results
        y_true.append(true_label)
        rb_pred.append(rb_label)
        gnb_pred.append(gnb_label)
        svm_pred.append(svm_label)
        lr_pred.append(lr_label)
        rf_pred.append(rf_label)
        mlp_pred.append(mlp_label)
        xgb_pred.append(xgb_label)
        image_ids.append(base_name)

        # Update progress bar with running accuracy
        # Running accuracy is computed incrementally as each image is processed
        n_done = len(y_true)
        if n_done > 0:
            rb_acc = (np.array(y_true) == np.array(rb_pred)).mean()
            gnb_acc = (np.array(y_true) == np.array(gnb_pred)).mean()
            svm_acc = (np.array(y_true) == np.array(svm_pred)).mean()
            lr_acc = (np.array(y_true) == np.array(lr_pred)).mean()
            rf_acc = (np.array(y_true) == np.array(rf_pred)).mean()
            mlp_acc = (np.array(y_true) == np.array(mlp_pred)).mean()
            xgb_acc = (np.array(y_true) == np.array(xgb_pred)).mean()
        else:
            rb_acc = gnb_acc = svm_acc = lr_acc = rf_acc = mlp_acc = xgb_acc = 0.0

        pbar.set_postfix({
            "done": f"{n_done}/{N}",
            "RB": f"{rb_acc:.3f}",
            "GNB": f"{gnb_acc:.3f}",
            "SVM": f"{svm_acc:.3f}",
            "LR": f"{lr_acc:.3f}",
            "RF": f"{rf_acc:.3f}",
            "MLP": f"{mlp_acc:.3f}",
            "XGB": f"{xgb_acc:.3f}"
        })

    except Exception as e:
        # Log failures for debugging
        fail_log.append((idx, str(e)))
        pbar.set_postfix({"error_at": idx})
        traceback.print_exc()
        continue

pbar.close()

# ============================================
# Final Evaluation Reports
# ============================================
if len(y_true) == 0:
    raise RuntimeError(
        "No samples collected. Check failure log and ground truth mapping. "
        "Verify that ground_truth_labels dictionary is properly populated."
    )

print("\n" + "=" * 60)
print("Classification Reports")
print("=" * 60)

print("\nRule-Based Classification Report:")
print(classification_report(
    y_true, rb_pred, labels=class_labels,
    target_names=target_names, digits=3, zero_division=0
))

print("\nGaussian Naive Bayes Classification Report:")
print(classification_report(
    y_true, gnb_pred, labels=class_labels,
    target_names=target_names, digits=3, zero_division=0
))

print("\nSupport Vector Machine Classification Report:")
print(classification_report(
    y_true, svm_pred, labels=class_labels,
    target_names=target_names, digits=3, zero_division=0
))

print("\nLogistic Regression Classification Report:")
print(classification_report(
    y_true, lr_pred, labels=class_labels,
    target_names=target_names, digits=3, zero_division=0
))

print("\nRandom Forest Classification Report:")
print(classification_report(
    y_true, rf_pred, labels=class_labels,
    target_names=target_names, digits=3, zero_division=0
))

print("\nMultilayer Perceptron Classification Report:")
print(classification_report(
    y_true, mlp_pred, labels=class_labels,
    target_names=target_names, digits=3, zero_division=0
))

print("\nXGBoost Classification Report:")
print(classification_report(
    y_true, xgb_pred, labels=class_labels,
    target_names=target_names, digits=3, zero_division=0
))

# ============================================
# Save Results to CSV
# ============================================
# Create comprehensive results DataFrame with all predictions
results_df = pd.DataFrame({
    'image': image_ids,
    'true_label': y_true,
    'true_name': [target_names[i] for i in y_true],
    'rb_pred': rb_pred,
    'rb_name': [target_names[i] for i in rb_pred],
    'gnb_pred': gnb_pred,
    'gnb_name': [target_names[i] for i in gnb_pred],
    'svm_pred': svm_pred,
    'svm_name': [target_names[i] for i in svm_pred],
    'lr_pred': lr_pred,
    'lr_name': [target_names[i] for i in lr_pred],
    'rf_pred': rf_pred,
    'rf_name': [target_names[i] for i in rf_pred],
    'mlp_pred': mlp_pred,
    'mlp_name': [target_names[i] for i in mlp_pred],
    'xgb_pred': xgb_pred,
    'xgb_name': [target_names[i] for i in xgb_pred],
})

csv_path = "rock_classification_comparison.csv"
results_df.to_csv(csv_path, index=False)
print(f"\n✓ Results saved to: {csv_path}")
print(f"  Total samples evaluated: {len(results_df)}")

# Save failure log if any errors occurred during evaluation
if fail_log:
    fail_df = pd.DataFrame(fail_log, columns=["idx", "error"])
    fail_path = "failed_images.csv"
    fail_df.to_csv(fail_path, index=False)
    print(f"⚠ {len(fail_log)} failed images logged to: {fail_path}")

print("\n" + "=" * 60)
print("Evaluation completed!")
print("=" * 60)


Evaluating 300 test images...


Evaluating images:   0%|          | 0/300 [00:00<?, ?img/s]


Classification Reports

Rule-Based Classification Report:
                  precision    recall  f1-score   support

 Ilmenite Basalt      0.887     0.925     0.905        93
  Olivine Basalt      0.753     0.688     0.719        80
Pigeonite Basalt      0.769     0.787     0.778       127

        accuracy                          0.803       300
       macro avg      0.803     0.800     0.801       300
    weighted avg      0.801     0.803     0.802       300


Gaussian Naive Bayes Classification Report:
                  precision    recall  f1-score   support

 Ilmenite Basalt      0.830     0.892     0.860        93
  Olivine Basalt      0.845     0.613     0.710        80
Pigeonite Basalt      0.754     0.843     0.796       127

        accuracy                          0.797       300
       macro avg      0.809     0.782     0.789       300
    weighted avg      0.802     0.797     0.793       300


Support Vector Machine Classification Report:
                  precision    